# 输入：让程序会"问问题"

---

**学习目标**

1. 掌握 `input()` 的三个特点
2. 记住最重要的一条：**`input()` 拿到的永远是字符串**
3. 会做类型转换，让输入变成能参与计算的数字
4. 知道 `eval(input())` 为什么危险，以及该用什么替代

**一句话理解**

`input()` 就像一个只会做笔录的书记员：
**你打进去什么，它就原封不动记成什么，一律当成"文字"。**


---
## 一、`input()` 的三个特点

**特点 1：会"卡住"程序，等用户输入**

程序执行到 `input()` 就会停下来，直到用户敲下回车，才继续往下走。

**特点 2：一般要把结果存进变量**

不存下来的话，用户输入的内容直接就丢掉了。

**特点 3（最重要）：返回的永远是字符串 `str`**

哪怕你输入 `123`，拿到的也是字符串 `'123'`，不是数字 `123`。


In [ ]:
# 特点 3 的验证（在 notebook 里会弹输入框，输入 123 试试）
value = input('请输入任意内容：')
print(f'你输入的是：{value!r}')          # !r 能看到它的"真面目"，字符串会带引号
print('它的类型是：', type(value))
print('它的长度是：', len(value))


**这就是为什么"输入两个数字求和"经常得到 `'12'` 而不是 `3`：**


In [ ]:
a = '1'
b = '2'
print("字符串相加：'1' + '2' =", a + b)      # 12 —— 这是拼接，不是相加！


---
## 二、让输入变成数字：三种方式

| 写法 | 得到什么 | 适用场景 | 风险 |
|---|---|---|---|
| `input()` | `str` | 姓名、地址、选项 | 无 |
| `int(input())` | `int` | 年龄、数量、整数 | 输入非数字会 `ValueError` |
| `float(input())` | `float` | 身高、价格、体重 | 输入非数字会 `ValueError` |

```python
name  = input('请输入你的名字：')          # 名字保持字符串就好
age   = int(input('请输入你的年龄：'))     # 年龄要数字
price = float(input('请输入单价：'))       # 价格要小数
```


In [ ]:
# 完整演示：做一个 BMI 计算器
name = input('请输入你的名字：')
height = float(input('请输入身高（米），例如 1.75：'))
weight = float(input('请输入体重（千克），例如 65：'))

bmi = weight / (height ** 2)
print(f'\n{name} 你好！')
print(f'你的 BMI 是 {bmi:.1f}')

# 顺手给一句健康提示，这就是"程序在和人对话"
if bmi < 18.5:
    print('偏瘦，注意补充营养')
elif bmi < 24:
    print('正常范围，保持住')
elif bmi < 28:
    print('偏胖，可以适当运动')
else:
    print('肥胖，建议咨询医生')


---
## 三、如果用户输入了非法内容怎么办？

`int('abc')` 会直接抛 `ValueError` 让程序崩溃。
生产环境里要**验证输入**，最基础的做法是用 `try/except` 兜住：


In [ ]:
def ask_int(prompt):
    """反复询问，直到用户输入一个合法整数"""
    while True:                      # 不合法就一直问
        raw = input(prompt)
        try:
            return int(raw)
        except ValueError:
            print(f'  "{raw}" 不是整数，请重新输入')


age = ask_int('请输入你的年龄：')
print(f'收到，你今年 {age} 岁，明年 {age + 1} 岁')


这个"一直问到对为止"的模式（`while True` + `return`）非常常用，
记住它的骨架，以后写任何交互程序都用得上。


---
## 四、【安全警告】`eval(input())` 绝对不要在真实项目里用

原始笔记里有这一行：

```python
X = eval(input("Please input number:"))   # 注释说"将字符串转换为整数"
```

**先纠正一个说法**：`eval` 不只是"转整数"，它是**把字符串当成 Python 代码来执行**。

```python
eval('1 + 2')        -> 3          （真的算了加法）
eval('"a" * 3')      -> 'aaa'      （真的重复了字符串）
eval('__import__("os").system("dir")')   -> 真的执行了系统命令！
```

**为什么危险？** 因为输入权在用户手里。如果用户输入的是
`__import__('os').system('删除所有文件')`，你的程序就真的会去执行它。

看一个"温和版"的演示，你能感受到它有多"听话"：


In [ ]:
user_input = '2 ** 10'          # 假设这是用户输入的内容
print('eval 直接把它当代码算了：', eval(user_input))

# 同样这段内容，用 int() 会怎样？
try:
    int('2 ** 10')
except ValueError as e:
    print('用 int() 转换失败：', e)
    print('但这是正确的行为 —— 只有纯数字才能转成整数')


### 正确替代方案

| 需求 | 不要用 | 应该用 |
|---|---|---|
| 转整数 | `eval(s)` | `int(s)` |
| 转小数 | `eval(s)` | `float(s)` |
| 解析用户给的列表/字典 | `eval(s)` | `json.loads(s)` 或 `ast.literal_eval(s)` |

`ast.literal_eval` 是"安全版的 eval"：它只允许字面量（数字、字符串、列表、字典等），
不允许函数调用，所以没法执行系统命令。


In [ ]:
import ast

print('安全解析列表：', ast.literal_eval('[1, 2, 3]'))
print('安全解析字典：', ast.literal_eval('{"a": 1}'))

try:
    ast.literal_eval('__import__("os").system("dir")')
except Exception as e:
    print(f'想执行命令？被拦下来了：{type(e).__name__}')


---
## 五、几个常用的小配合

### 5.1 查关键字（写变量名时避免撞车）


In [ ]:
import keyword

print('共有', len(keyword.kwlist), '个关键字')
print(keyword.kwlist)
print('is 是关键字吗？', keyword.iskeyword('is'))


### 5.2 配合条件判断做"登录校验"


In [ ]:
def login_demo(username, password):
    # 真实项目里密码不会明文比较，这里只是演示逻辑
    if username == 'admin' and password == '123456':
        return '登录成功'
    elif username != 'admin':
        return '用户名不存在'
    else:
        return '密码错误'


print(login_demo('admin', '123456'))
print(login_demo('admin', 'wrong'))


### 5.3 随机数：给程序加点"意外"

`random` 是标准库，不用安装。三个最常用的函数：


In [ ]:
import random

print('randint(1, 10)      ->', random.randint(1, 10))       # 闭区间 [1, 10] 的整数
print('randrange(1, 10)    ->', random.randrange(1, 10))     # 左闭右开 [1, 10)
print('choice 从列表里挑一个 ->', random.choice(['石头', '剪刀', '布']))
print('random()            ->', round(random.random(), 4))   # [0, 1) 的浮点数

# 想每次运行结果一样（比如做实验对比），可以固定随机种子
random.seed(42)
print('固定种子后第一次：', random.randint(1, 100))
random.seed(42)
print('重新设成同一个种子：', random.randint(1, 100))       # 和上一行相同


---
## 六、常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 忘了 `input()` 返回字符串 | `'1' + '2'` 得到 `'12'` | 要计算先 `int()` / `float()` |
| `int()` 直接套 `input()` 没做校验 | 用户输错就崩 | 用 `try/except` + 循环重问 |
| 用 `eval(input())` | 有执行任意代码的安全风险 | `int()` / `float()` / `ast.literal_eval()` |
| 忘了写提示语 | 用户不知道要输什么 | `input('请输入年龄：')` |
| `input()` 里写了很多字 | 提示太长反而看不清 | 提示语简短，说明放 print 里 |
| 在 notebook 里调试忘了有输入 | 单元格一直转圈 | 记得点输入框并回车 |
| 提示语里加了 `\n` | 换行后光标位置奇怪 | 需要换行就用 `print()` 先打一行 |


---
## 七、动手练习

**练习 1**：写一个"求两数之和"的小程序，要求分别输入两个数，输出它们的和与差。

**练习 2**：写一个程序，输入圆的半径，输出圆的周长和面积（保留 2 位小数）。

**练习 3**：写一个"猜数字"游戏：
- 程序随机生成 1~100 的整数
- 循环让用户猜，提示"猜大了 / 猜小了"
- 猜中后输出总共猜了几次

**练习 4**：把上面的 `ask_int()` 改造成 `ask_float()`，并要求输入值必须大于 0。

**练习 5**：解释下面这段代码为什么"永远进不去 if 分支"，并改对：
```python
age = input('请输入年龄：')
if age > 18:
    print('成年了')
```
提示：字符串和整数比较会抛 `TypeError`。

---

**练习 3 参考答案（先自己做）**

```python
import random

target = random.randint(1, 100)
count = 0
while True:
    count += 1
    guess = int(input('猜一个 1~100 的数：'))
    if guess == target:
        print(f'猜对了！你一共猜了 {count} 次')
        break
    print('猜大了' if guess > target else '猜小了')
```
